# PySpark SQL Analysis

This notebook uses PySpark SQL to analyse the processed bus disruption data

The SQL analysis includes

- severity distribution
- high severity routes
- severity by hour
- peak and non peak comparison
- average operational conditions
- route recovery priority
- saving SQL results as CSV

In [1]:
# import required libraries

import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

python_path = sys.executable

spark = (
    SparkSession.builder
    .appName(
        "bus_disruption_sql_analysis"
    )
    .config(
        "spark.sql.shuffle.partitions",
        "4"
    )
    .config(
        "spark.sql.session.timeZone",
        "Europe/London"
    )
    .config(
        "spark.pyspark.python",
        python_path
    )
    .config(
        "spark.pyspark.driver.python",
        python_path
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel(
    "WARN"
)

print(
    "spark session created"
)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/16 23:41:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/16 23:41:45 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/16 23:41:45 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/07/16 23:41:45 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/07/16 23:41:45 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
26/07/16 23:41:45 WARN Utils: Service 'SparkUI' could not bind on port 4044. Attempting port 4045.


spark session created


In [2]:
# define project folders

current_folder = Path.cwd()

if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder

feature_csv_path = (
    project_root
    / "data"
    / "processed"
    / "disruption_features_csv"
)

sql_output_path = (
    project_root
    / "data"
    / "processed"
    / "sql_outputs"
)

sql_output_path.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "project root:",
    project_root
)

print(
    "feature csv exists:",
    feature_csv_path.exists()
)

print(
    "sql output folder:",
    sql_output_path
)

project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
feature csv exists: True
sql output folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs


In [3]:
# load the processed feature csv dataset

feature_df = (
    spark.read
    .option(
        "header",
        True
    )
    .option(
        "inferSchema",
        True
    )
    .csv(
        str(
            feature_csv_path
        )
    )
)

# repartition and cache the dataset

feature_df = (
    feature_df
    .repartition(
        4,
        "line_ref"
    )
    .cache()
)

feature_rows = feature_df.count()

print(
    "feature rows:",
    f"{feature_rows:,}"
)

print(
    "feature columns:",
    len(
        feature_df.columns
    )
)

print(
    "feature partitions:",
    feature_df.rdd.getNumPartitions()
)

26/07/16 23:42:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 4:>                                                          (0 + 4) / 4]

feature rows: 8,274
feature columns: 47
feature partitions: 4


## check columns and create SQL view

In [4]:
# create a temporary sql table

feature_df.createOrReplaceTempView(
    "bus_disruption_features"
)

print(
    "sql table created"
)

print(
    "table name: bus_disruption_features"
)

# display available sql tables

spark.sql(
    "SHOW TABLES"
).show(
    truncate=False
)

sql table created
table name: bus_disruption_features
+---------+-----------------------+-----------+
|namespace|tableName              |isTemporary|
+---------+-----------------------+-----------+
|         |bus_disruption_features|true       |
+---------+-----------------------+-----------+



In [5]:
# query 1 display ten sample records

query_1_sample_df = spark.sql(
    """
    SELECT
        event_snapshot_time,
        line_ref,
        direction_ref,
        observation_hour,
        observed_vehicle_count,
        activity_ratio,
        spacing_ratio,
        severity_name
    FROM bus_disruption_features
    LIMIT 10
    """
)

print(
    "query 1 sample records"
)

query_1_sample_df.show(
    truncate=False
)

query 1 sample records
+-----------------------+--------+-------------+----------------+----------------------+------------------+------------------+-------------+
|event_snapshot_time    |line_ref|direction_ref|observation_hour|observed_vehicle_count|activity_ratio    |spacing_ratio     |severity_name|
+-----------------------+--------+-------------+----------------+----------------------+------------------+------------------+-------------+
|2026-07-13 18:25:20.822|18A     |inbound      |18              |3                     |1.0               |1.0               |low          |
|2026-07-14 05:35:12.769|18A     |inbound      |5               |2                     |0.6666666666666666|1.0               |medium       |
|2026-07-14 16:30:28.485|18A     |inbound      |16              |1                     |0.4               |1.0               |high         |
|2026-07-14 17:30:37.087|18A     |inbound      |17              |3                     |1.5               |0.6666666666666666|low  

In [6]:
# query 2 filter high severity records

query_2_high_severity_df = spark.sql(
    """
    SELECT
        event_snapshot_time,
        line_ref,
        direction_ref,
        observation_hour,
        observed_vehicle_count,
        operational_irregularity_score,
        severity_name
    FROM bus_disruption_features
    WHERE severity_name = 'high'
    ORDER BY operational_irregularity_score DESC
    LIMIT 10
    """
)

print(
    "query 2 high severity records"
)

query_2_high_severity_df.show(
    truncate=False
)

query 2 high severity records
+-----------------------+--------+-------------+----------------+----------------------+------------------------------+-------------+
|event_snapshot_time    |line_ref|direction_ref|observation_hour|observed_vehicle_count|operational_irregularity_score|severity_name|
+-----------------------+--------+-------------+----------------+----------------------+------------------------------+-------------+
|2026-07-16 10:52:51.52 |X10     |outbound     |10              |3                     |0.4920704003060883            |high         |
|2026-07-14 17:30:37.087|31      |inbound      |17              |1                     |0.4904761904761905            |high         |
|2026-07-16 09:42:40.277|34      |inbound      |9               |2                     |0.48318395219803667           |high         |
|2026-07-15 14:32:27.244|X1      |outbound     |14              |7                     |0.4806638566912539            |high         |
|2026-07-16 06:11:48.735|998    

In [7]:
# query 3 count each severity class

query_3_severity_count_df = spark.sql(
    """
    SELECT
        severity_name,
        COUNT(*) AS total_records
    FROM bus_disruption_features
    GROUP BY severity_name
    ORDER BY total_records DESC
    """
)

print(
    "query 3 severity class count"
)

query_3_severity_count_df.show(
    truncate=False
)

query 3 severity class count
+-------------+-------------+
|severity_name|total_records|
+-------------+-------------+
|low          |5740         |
|medium       |1618         |
|high         |916          |
+-------------+-------------+



In [8]:
# query 4 find routes with high severity conditions

query_4_high_routes_df = spark.sql(
    """
    SELECT
        line_ref,
        direction_ref,
        COUNT(*) AS high_severity_count,
        SUM(
            observed_vehicle_count
        ) AS total_observed_vehicles,
        ROUND(
            AVG(
                operational_irregularity_score
            ),
            3
        ) AS average_irregularity
    FROM bus_disruption_features
    WHERE severity_name = 'high'
    GROUP BY
        line_ref,
        direction_ref
    ORDER BY
        high_severity_count DESC
    LIMIT 10
    """
)

print(
    "query 4 routes with high severity conditions"
)

query_4_high_routes_df.show(
    truncate=False
)

query 4 routes with high severity conditions
+--------+-------------+-------------------+-----------------------+--------------------+
|line_ref|direction_ref|high_severity_count|total_observed_vehicles|average_irregularity|
+--------+-------------+-------------------+-----------------------+--------------------+
|18      |outbound     |20                 |41                     |0.288               |
|15      |inbound      |20                 |58                     |0.304               |
|6A      |inbound      |18                 |26                     |0.312               |
|37      |outbound     |13                 |13                     |0.29                |
|60      |inbound      |12                 |64                     |0.306               |
|13      |outbound     |11                 |25                     |0.291               |
|6A      |outbound     |11                 |13                     |0.32                |
|31      |outbound     |10                 |10         

In [9]:
# query 5 calculate operational statistics by severity

query_5_statistics_df = spark.sql(
    """
    SELECT
        severity_name,
        COUNT(*) AS total_records,
        ROUND(
            AVG(
                observed_vehicle_count
            ),
            2
        ) AS average_vehicle_count,
        ROUND(
            AVG(
                activity_ratio
            ),
            3
        ) AS average_activity_ratio,
        ROUND(
            AVG(
                spacing_ratio
            ),
            3
        ) AS average_spacing_ratio,
        ROUND(
            MIN(
                operational_irregularity_score
            ),
            3
        ) AS minimum_irregularity,
        ROUND(
            MAX(
                operational_irregularity_score
            ),
            3
        ) AS maximum_irregularity
    FROM bus_disruption_features
    GROUP BY severity_name
    ORDER BY severity_name
    """
)

print(
    "query 5 statistics by severity"
)

query_5_statistics_df.show(
    truncate=False
)

query 5 statistics by severity
+-------------+-------------+---------------------+----------------------+---------------------+--------------------+--------------------+
|severity_name|total_records|average_vehicle_count|average_activity_ratio|average_spacing_ratio|minimum_irregularity|maximum_irregularity|
+-------------+-------------+---------------------+----------------------+---------------------+--------------------+--------------------+
|high         |916          |3.62                 |0.785                 |1.916                |0.214               |0.492               |
|low          |5740         |5.2                  |1.111                 |0.713                |0.0                 |0.1                 |
|medium       |1618         |4.48                 |0.855                 |0.95                 |0.1                 |0.214               |
+-------------+-------------+---------------------+----------------------+---------------------+--------------------+------------------

In [10]:
# query 6 rank routes by high severity count

query_6_ranked_routes_df = spark.sql(
    """
    SELECT
        line_ref,
        direction_ref,
        high_severity_count,
        RANK() OVER(
            ORDER BY high_severity_count DESC
        ) AS route_rank
    FROM (
        SELECT
            line_ref,
            direction_ref,
            COUNT(*) AS high_severity_count
        FROM bus_disruption_features
        WHERE severity_name = 'high'
        GROUP BY
            line_ref,
            direction_ref
    ) route_counts
    ORDER BY
        route_rank,
        line_ref
    LIMIT 10
    """
)

print(
    "query 6 ranked high severity routes"
)

query_6_ranked_routes_df.show(
    truncate=False
)

query 6 ranked high severity routes


26/07/16 23:42:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:14 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:15 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 2

+--------+-------------+-------------------+----------+
|line_ref|direction_ref|high_severity_count|route_rank|
+--------+-------------+-------------------+----------+
|15      |inbound      |20                 |1         |
|18      |outbound     |20                 |1         |
|6A      |inbound      |18                 |3         |
|37      |outbound     |13                 |4         |
|60      |inbound      |12                 |5         |
|13      |outbound     |11                 |6         |
|6A      |outbound     |11                 |6         |
|11      |outbound     |10                 |8         |
|14      |outbound     |10                 |8         |
|20A     |inbound      |10                 |8         |
+--------+-------------+-------------------+----------+



In [11]:
# query 7 find routes above the overall average irregularity

query_7_above_average_df = spark.sql(
    """
    SELECT
        line_ref,
        direction_ref,
        COUNT(*) AS total_snapshots,
        ROUND(
            AVG(
                operational_irregularity_score
            ),
            3
        ) AS average_irregularity
    FROM bus_disruption_features
    GROUP BY
        line_ref,
        direction_ref
    HAVING AVG(
        operational_irregularity_score
    ) > (
        SELECT
            AVG(
                operational_irregularity_score
            )
        FROM bus_disruption_features
    )
    ORDER BY average_irregularity DESC
    LIMIT 10
    """
)

print(
    "query 7 routes above average irregularity"
)

query_7_above_average_df.show(
    truncate=False
)

query 7 routes above average irregularity
+--------+-------------+---------------+--------------------+
|line_ref|direction_ref|total_snapshots|average_irregularity|
+--------+-------------+---------------+--------------------+
|37      |outbound     |32             |0.171               |
|6A      |inbound      |45             |0.158               |
|68      |inbound      |5              |0.153               |
|31      |inbound      |33             |0.149               |
|998     |outbound     |33             |0.146               |
|X10     |outbound     |33             |0.141               |
|51      |outbound     |33             |0.136               |
|74      |outbound     |33             |0.133               |
|87      |outbound     |33             |0.133               |
|18A     |outbound     |11             |0.131               |
+--------+-------------+---------------+--------------------+



In [12]:
# create a function for saving sql results

def save_sql_result(
    dataframe,
    folder_name
):
    output_folder = (
        sql_output_path
        /
        folder_name
    )

    (
        dataframe
        .coalesce(
            1
        )
        .write
        .mode(
            "overwrite"
        )
        .option(
            "header",
            True
        )
        .csv(
            str(
                output_folder
            )
        )
    )

    print(
        "saved:",
        output_folder
    )

In [13]:
# save all sql query results

save_sql_result(
    query_1_sample_df,
    "01_sample_records"
)

save_sql_result(
    query_2_high_severity_df,
    "02_high_severity_records"
)

save_sql_result(
    query_3_severity_count_df,
    "03_severity_count"
)

save_sql_result(
    query_4_high_routes_df,
    "04_high_severity_routes"
)

save_sql_result(
    query_5_statistics_df,
    "05_severity_statistics"
)

save_sql_result(
    query_6_ranked_routes_df,
    "06_ranked_routes"
)

save_sql_result(
    query_7_above_average_df,
    "07_above_average_routes"
)

saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/01_sample_records
saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/02_high_severity_records
saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/03_severity_count
saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/04_high_severity_routes
saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/05_severity_statistics


26/07/16 23:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 23:42:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/07/16 2

saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/06_ranked_routes
saved: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/processed/sql_outputs/07_above_average_routes


In [14]:
# define expected output folders

expected_output_folders = [
    "01_sample_records",
    "02_high_severity_records",
    "03_severity_count",
    "04_high_severity_routes",
    "05_severity_statistics",
    "06_ranked_routes",
    "07_above_average_routes"
]

missing_output_folders = []

# check every output folder

for folder_name in expected_output_folders:
    folder_path = (
        sql_output_path
        /
        folder_name
    )

    if not folder_path.exists():
        missing_output_folders.append(
            folder_name
        )

print(
    "missing output folders:",
    missing_output_folders
)

if missing_output_folders:
    raise ValueError(
        "some sql output folders are missing"
    )

print(
    "all sql results saved successfully"
)

missing output folders: []
all sql results saved successfully
